In [1]:
!pip install transformers datasets peft accelerate bitsandbytes --quiet

In [2]:
pip install -U -q transformers datasets peft accelerate

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install -U -q bitsandbytes

In [2]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TrainingArguments, Trainer, default_data_collator
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import time
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re
import bitsandbytes

2025-12-19 09:11:10.132994: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766135470.324998      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766135470.381427      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766135470.859648      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766135470.859688      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766135470.859691      55 computation_placer.cc:177] computation placer alr

ModuleNotFoundError: No module named 'bitsandbytes'

In [5]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("YOUR_GROQ_KEY")


client = OpenAI(
  api_key= api_key,
  base_url="https://api.groq.com/openai/v1"
)

In [6]:
SOURCE_TEXT = """
آقامحمدخان قاجار

آقامحمدخان قاجار بنیان‌گذار سلسله قاجار بود و در دوره‌ای به قدرت رسید که ایران پس از سقوط صفویه و حکومت‌های کوتاه‌مدت افشاریه و زندیه دچار هرج‌ومرج، تجزیه سیاسی و ضعف اقتدار مرکزی شده بود. او از ایل قاجار و شاخه قاجار قَوَنلو بود و از همان ابتدای زندگی، به دلیل شرایط سخت و خشونت‌آمیز دوران، شخصیتی سخت‌گیر، بی‌رحم و کاملاً نظامی پیدا کرد.

هدف اصلی آقامحمدخان، نه اصلاحات اجتماعی یا اقتصادی، بلکه بازسازی اقتدار سیاسی و تمامیت ارضی ایران بود. او با لشکرکشی‌های گسترده، بسیاری از مناطق جداشده از مرکز را دوباره تحت کنترل حکومت مرکزی درآورد. در این مسیر، برخوردهای بسیار خشن با مخالفان داشت که باعث ایجاد ترس و اطاعت، اما نه وفاداری، در میان مردم شد.

آقامحمدخان تهران را به‌عنوان پایتخت انتخاب کرد؛ اقدامی که اهمیت راهبردی داشت و تا امروز نیز ادامه یافته است. با این حال، در دوران کوتاه سلطنت او، هیچ برنامه‌ای برای بهبود وضعیت معیشتی مردم، توسعه اقتصادی یا اصلاح ساختار اداری کشور دیده نمی‌شود. حکومت او بیشتر شبیه یک دولت نظامی متمرکز بود که هدفش تثبیت قدرت بود، نه توسعه.

قتل آقامحمدخان در قفقاز به دست خادمانش، نشان‌دهنده فضای پرتنش و ناامن درونی حکومت اوست. با وجود خشونت‌های فراوان، نقش او در یکپارچه‌سازی ایران غیرقابل انکار است.

فتحعلی‌شاه قاجار

فتحعلی‌شاه پس از آقامحمدخان به سلطنت رسید و برخلاف او، پادشاهی تجمل‌گرا، درباری و کم‌توجه به امور نظامی بود. دوران سلطنت او هم‌زمان با گسترش نفوذ قدرت‌های استعماری روسیه و انگلستان در منطقه بود؛ مسئله‌ای که ایران آمادگی سیاسی و نظامی لازم برای مقابله با آن را نداشت.

مهم‌ترین رویدادهای دوره فتحعلی‌شاه، جنگ‌های ایران و روسیه بود که به شکست ایران انجامید. نتیجه این جنگ‌ها، امضای قراردادهای گلستان و ترکمانچای بود که بخش‌های وسیعی از سرزمین‌های قفقاز از ایران جدا شد. این قراردادها نه‌تنها ضربه‌ای بزرگ به تمامیت ارضی ایران وارد کرد، بلکه باعث تضعیف روحیه ملی و افزایش نفوذ بیگانگان شد.

فتحعلی‌شاه به جای اصلاح ساختار ارتش و حکومت، بیشتر به تثبیت قدرت خاندان خود و گسترش حرمسرا و تشریفات سلطنتی پرداخت. وابستگی مالی دربار به وام‌ها و کمک‌های خارجی افزایش یافت و شکاف میان حکومت و مردم عمیق‌تر شد.

در این دوره، ایران عملاً وارد مرحله‌ای از وابستگی سیاسی و نظامی شد که پیامدهای آن تا سال‌ها بعد ادامه یافت.

محمدشاه قاجار

محمدشاه قاجار در شرایطی به قدرت رسید که ضعف ساختاری حکومت قاجار آشکار شده بود. او تلاش‌هایی برای تقویت اقتدار مرکزی انجام داد، اما بیماری، ضعف اراده و وابستگی شدید به اطرافیان و صدراعظم‌ها، مانع از موفقیت او شد.

در دوران محمدشاه، نفوذ انگلستان در سیاست ایران افزایش یافت و تلاش ایران برای بازپس‌گیری هرات با فشار خارجی ناکام ماند. این مسئله نشان داد که ایران در تصمیمات کلان سیاست خارجی، استقلال چندانی ندارد.

اقتصاد کشور در این دوره همچنان سنتی، ناکارآمد و وابسته به مالیات‌های سنگین بود. نارضایتی اجتماعی افزایش یافت، اما هنوز جنبش سیاسی سازمان‌یافته‌ای شکل نگرفته بود. محمدشاه بیشتر نماد دولت ضعیف در برابر قدرت‌های خارجی است.

ناصرالدین‌شاه قاجار

ناصرالدین‌شاه یکی از مهم‌ترین و تأثیرگذارترین شاهان تاریخ معاصر ایران است. سلطنت طولانی او هم‌زمان با آغاز آشنایی جدی ایران با تمدن غرب بود. او چندین بار به اروپا سفر کرد و با مظاهر پیشرفت غرب آشنا شد، اما این آشنایی بیشتر جنبه ظاهری داشت و به اصلاحات عمیق ساختاری منجر نشد.

در دوره او، امتیازات اقتصادی متعددی به خارجی‌ها داده شد که مهم‌ترین آن‌ها امتیاز تنباکو بود. این اقدام با واکنش شدید مردم و روحانیون مواجه شد و جنبش تحریم تنباکو به‌عنوان نخستین حرکت گسترده سیاسی-مذهبی در تاریخ معاصر ایران شکل گرفت.

ناصرالدین‌شاه تلاش می‌کرد قدرت مطلقه خود را حفظ کند، اما جامعه به‌تدریج آگاه‌تر شده بود. مطبوعات، مدارس جدید و ارتباط با غرب، زمینه‌های بیداری فکری را فراهم کرد. ترور ناصرالدین‌شاه را می‌توان نشانه‌ای از پایان دوران سلطنت مطلقه سنتی دانست.

مظفرالدین‌شاه قاجار

مظفرالدین‌شاه پادشاهی ضعیف، بیمار و وابسته بود، اما نام او با یکی از مهم‌ترین رویدادهای تاریخ ایران گره خورده است: انقلاب مشروطه. فشارهای اقتصادی، وام‌های خارجی و نارضایتی عمومی باعث شکل‌گیری جنبش مشروطه شد.

او سرانجام فرمان مشروطه را امضا کرد و مجلس شورای ملی تشکیل شد. این اتفاق نقطه عطفی در تاریخ سیاسی ایران بود، زیرا برای نخستین بار قدرت شاه محدود شد و قانون جایگاه رسمی پیدا کرد.

با این حال، ضعف مدیریتی مظفرالدین‌شاه مانع از تثبیت کامل نظام مشروطه شد و مشکلات ساختاری همچنان باقی ماند.

محمدعلی‌شاه قاجار

محمدعلی‌شاه برخلاف پدرش، مخالف مشروطه بود و تلاش کرد سلطنت مطلقه را احیا کند. او مجلس را به توپ بست و بسیاری از مشروطه‌خواهان را سرکوب کرد.

اما مقاومت نیروهای مردمی و مشروطه‌خواهان در شهرهای مختلف، در نهایت به سقوط او انجامید. شکست محمدعلی‌شاه نشان داد که جامعه ایران دیگر سلطنت مطلقه را به‌راحتی نمی‌پذیرد.

احمدشاه قاجار

احمدشاه آخرین شاه قاجار بود و در دوره‌ای حکومت کرد که ایران با بحران‌های شدید سیاسی، اقتصادی و امنیتی روبه‌رو بود. ضعف دولت مرکزی، نفوذ قدرت‌های خارجی و ناتوانی شاه در اداره کشور، شرایط را برای ظهور رضاخان فراهم کرد.

احمدشاه بیشتر نظاره‌گر تحولات بود تا بازیگر اصلی آن‌ها. در نهایت، با برکناری او، سلسله قاجار پایان یافت.

رضاشاه پهلوی

رضاشاه با تکیه بر ارتش به قدرت رسید و دولت متمرکز مدرن را در ایران بنیان گذاشت. او اصلاحات گسترده‌ای در ارتش، آموزش، زیرساخت و ادارات دولتی انجام داد و تلاش کرد ایران را به یک کشور مدرن تبدیل کند.

در عین حال، حکومت او استبدادی بود و آزادی‌های سیاسی به‌شدت محدود شد. برخورد سخت با روحانیت و مخالفان سیاسی، نارضایتی‌هایی ایجاد کرد که در آینده اثرگذار بود.

محمدرضاشاه پهلوی

محمدرضاشاه پهلوی در فضایی پیچیده از جنگ سرد، وابستگی خارجی و تحولات اجتماعی حکومت کرد. اصلاحات اقتصادی و اجتماعی او، به‌ویژه انقلاب سفید، ساختار جامعه را تغییر داد، اما هم‌زمان شکاف طبقاتی، سرکوب سیاسی و بحران هویت فرهنگی را تشدید کرد.

در نهایت، مجموعه‌ای از نارضایتی‌های سیاسی، اقتصادی و فرهنگی به انقلاب اسلامی ۱۳۵۷ انجامید و سلطنت در ایران پایان یافت.
دلایل انقلاب

انقلاب اسلامی محصول مجموعه‌ای از عوامل داخلی و خارجی بود. در سطح داخلی، سیاست‌های پهلوی باعث ایجاد نابرابری‌های اقتصادی و اجتماعی شد. توسعه صنعتی و شهری با تمرکز بر کلان‌شهرها همراه بود و بخش‌های روستایی و محروم جامعه از رشد اقتصادی بهره چندانی نبردند. فساد مالی و سیاسی نیز در سطوح مختلف حکومت مشاهده می‌شد و اعتماد مردم به نهادهای حکومتی کاهش یافته بود.

در بعد سیاسی، محدودیت آزادی‌های اجتماعی و سرکوب مخالفان سیاسی باعث شد که گروه‌های مختلف سیاسی، مذهبی و اجتماعی به تدریج در برابر حکومت متحد شوند. رسانه‌ها تحت کنترل شدید دولت بودند و امکان بیان آزادانه نقدها وجود نداشت. این وضعیت، به همراه فقدان ساختارهای قانونی و پارلمانی قوی، نارضایتی‌ها را تشدید کرد.

از منظر فرهنگی و مذهبی، بخشی از جامعه ایران که ارزش‌های سنتی و مذهبی را مهم می‌دانست، نسبت به روند مدرنیزاسیون و غرب‌گرایی پهلوی واکنش نشان داد. رشد آموزش‌های نوین و ارتباطات با فرهنگ غربی باعث شد که شکاف بین سنت و مدرنیته در جامعه عمیق‌تر شود و زمینه برای ظهور جنبش‌های مذهبی فراهم گردد.

نقش اقشار مختلف جامعه

یکی از ویژگی‌های مهم انقلاب اسلامی، مشارکت گسترده مردم از اقشار مختلف جامعه بود. دانشجویان، روشنفکران، روحانیون، کارگران و حتی برخی از طبقات متوسط شهری در تظاهرات و فعالیت‌های سیاسی شرکت کردند. روحانیون به ویژه نقش رهبری و سازماندهی اعتراضات را بر عهده داشتند و پیامدهای انقلاب را به سمت ایجاد نظامی مبتنی بر ارزش‌های اسلامی هدایت کردند.

زنان و جوانان نیز به شکل فعال در تحولات سیاسی و اجتماعی مشارکت داشتند. گرچه قوانین و محدودیت‌های سنتی در برخی موارد حضور آنان را محدود می‌کرد، اما نقش آن‌ها در تبلیغات، تظاهرات و فعالیت‌های اجتماعی تأثیرگذار بود. این مشارکت گسترده باعث شد که انقلاب به یک حرکت ملی تبدیل شود و حمایت بخش‌های مختلف جامعه را به دست آورد.

پیامدهای سیاسی و اجتماعی

انقلاب اسلامی در نهایت منجر به سقوط سلطنت پهلوی و تشکیل جمهوری اسلامی شد. تغییرات گسترده سیاسی شامل تأسیس نهادهای حکومتی جدید، قانون اساسی مبتنی بر اصول اسلامی و شکل‌گیری مجلس شورای اسلامی بود. این تحولات به بازتعریف قدرت سیاسی و ایجاد ساختارهای قانونی و اداری تازه منجر شد.

از منظر اجتماعی، انقلاب باعث افزایش آگاهی سیاسی مردم و شکل‌گیری جنبش‌های مدنی و مذهبی شد. در عین حال، تغییرات سریع سیاسی و اجتماعی چالش‌هایی نیز به همراه داشت، از جمله تنش‌های فرهنگی، محدودیت‌های آزادی‌های فردی و ضرورت بازسازی اقتصادی پس از انقلاب.
پس از پیروزی انقلاب اسلامی در سال ۱۳۵۷ و تشکیل جمهوری اسلامی، ایران وارد مرحله‌ای جدید از تاریخ خود شد که با تحولات سیاسی، اجتماعی و اقتصادی گسترده همراه بود. این دوره به ویژه با جنگ تحمیلی، بازسازی کشور، اصلاحات اقتصادی و تغییرات فرهنگی شناخته می‌شود.

جنگ تحمیلی و تأثیرات آن

یکی از مهم‌ترین رخدادهای دهه اول پس از انقلاب، جنگ تحمیلی ایران و عراق (۱۳۵۹–۱۳۶۷) بود. این جنگ هشت ساله تأثیرات عمیقی بر جامعه، اقتصاد و سیاست داخلی ایران داشت. از لحاظ سیاسی، جنگ باعث شد که تمرکز قدرت در دست دولت و رهبری انقلاب مستحکم شود و نهادهای نظامی و امنیتی اهمیت بیشتری پیدا کنند. از منظر اجتماعی، جامعه ایران با بحران‌های انسانی و جابجایی‌های گسترده مواجه شد. خانواده‌ها، شهروندان و نیروهای داوطلب نقش فعال و فداکارانه‌ای در پشتیبانی از جبهه‌ها ایفا کردند، که این امر همبستگی ملی و حس مسئولیت اجتماعی را تقویت کرد.

اقتصاد ایران در این دوران با چالش‌های بزرگی روبرو بود. بخش عمده منابع کشور به تأمین نیازهای جنگ اختصاص یافت و تولید داخلی کاهش یافت. تلفات انسانی و زیرساخت‌های تخریب‌شده، نیاز به بازسازی گسترده کشور پس از پایان جنگ را ضروری ساخت. این دوره همچنین باعث ایجاد تجربه مدیریتی و اقتصادی جدید شد که در سال‌های بعد برای برنامه‌ریزی توسعه کشور مورد استفاده قرار گرفت.

اصلاحات اقتصادی و اجتماعی

پس از پایان جنگ، دولت جمهوری اسلامی به دنبال بازسازی کشور و توسعه اقتصادی بود. برنامه‌های بازسازی و اصلاحات اقتصادی شامل سرمایه‌گذاری در زیرساخت‌ها، صنایع و آموزش بود. توسعه نظام بانکی، تقویت بخش کشاورزی و افزایش تولید داخلی از جمله اقدامات اصلی بود. با وجود این، مشکلاتی مانند تورم، تحریم‌های خارجی و محدودیت‌های داخلی چالش‌های اساسی اقتصادی ایران باقی ماند.

از نظر اجتماعی، این دوره با تغییرات فرهنگی و ارتقای سطح آموزش و پرورش همراه بود. دولت تلاش کرد دسترسی به آموزش و خدمات اجتماعی را برای همه اقشار جامعه فراهم کند. همچنین توجه به حقوق زنان و کودکان در قوانین و برنامه‌های اجتماعی افزایش یافت، هرچند محدودیت‌ها و سنت‌های موجود گاهی مانع از تحقق کامل این اهداف می‌شد.

تحولات فرهنگی و سیاسی

در عرصه فرهنگی، دوران جمهوری اسلامی با تأکید بر ارزش‌های اسلامی و بازتعریف هویت ملی همراه بود. نهادهای فرهنگی، رسانه‌ها و نظام آموزشی به گونه‌ای طراحی شدند که ارزش‌های انقلاب و اصول اسلامی را منتقل کنند. در عین حال، فعالیت‌های هنری و رسانه‌ای نیز به مرور متنوع‌تر شد و بخشی از جامعه به دنبال بیان دیدگاه‌های نوین فرهنگی و اجتماعی بود.

از منظر سیاسی، نهادهای حکومتی و مجلس شورای اسلامی شکل گرفتند و قانون اساسی جمهوری اسلامی به عنوان چارچوب کلی نظام سیاسی تدوین شد. این ساختارها باعث ایجاد ثبات نسبی سیاسی در کشور شدند، اما همزمان با ظهور گروه‌ها و جریان‌های مخالف داخلی و فشارهای بین‌المللی، چالش‌هایی در سیاست داخلی و خارجی ایران ایجاد شد.
ایران در دهه‌های اخیر با تغییرات سریع اقتصادی، اجتماعی، سیاسی و فرهنگی مواجه بوده است که بازتاب‌دهنده تحولات گسترده در تاریخ معاصر کشور است. این دوره، با رشد جمعیت، پیشرفت فناوری، جهانی شدن و فشارهای بین‌المللی همراه بوده و چالش‌های جدیدی برای جامعه ایرانی ایجاد کرده است.

اقتصاد و سیاست داخلی

اقتصاد ایران در دوره معاصر با فرصت‌ها و تهدیدهای مختلفی روبروست. منابع طبیعی فراوان، به ویژه نفت و گاز، یکی از پایه‌های اصلی اقتصاد کشور هستند، اما وابستگی به درآمدهای نفتی باعث شده است که اقتصاد ایران به نوسانات بازار جهانی حساس باشد. علاوه بر این، تحریم‌های بین‌المللی و محدودیت‌های تجاری، چالش‌های اقتصادی جدی ایجاد کرده‌اند. در کنار این مسائل، نابرابری‌های منطقه‌ای و اجتماعی، اشتغال و تورم از مهم‌ترین دغدغه‌های مردم به شمار می‌روند.

سیاست داخلی ایران نیز در این دوره دچار تحولات و رقابت‌های مختلف شده است. نهادهای انتخابی و حکومتی با هم تعامل دارند و گروه‌های مختلف سیاسی، اجتماعی و فرهنگی برای تأثیرگذاری بر سیاست‌ها و برنامه‌های کشور فعالیت می‌کنند. مشارکت مردم در انتخابات و اعتراض‌های مدنی، نشان‌دهنده اهمیت نقش جامعه در تعیین مسیر سیاسی کشور است. با این حال، وجود محدودیت‌ها و فشارهای امنیتی، گاهی فضای آزادی عمل نهادها و شهروندان را محدود کرده است.

روابط خارجی و منطقه‌ای

ایران در عرصه بین‌المللی با چالش‌ها و فرصت‌های مختلفی مواجه است. موقعیت جغرافیایی و منابع طبیعی کشور باعث شده که ایران در سیاست‌های منطقه‌ای و جهانی نقش مهمی داشته باشد. روابط با کشورهای همسایه، قدرت‌های جهانی و سازمان‌های بین‌المللی همواره بر اقتصاد، امنیت و سیاست داخلی تأثیرگذار بوده است. چالش‌های امنیتی، رقابت‌های منطقه‌ای و تحریم‌های بین‌المللی از جمله عوامل تعیین‌کننده در سیاست خارجی ایران به شمار می‌روند.

جوانان و تغییرات اجتماعی

جوانان ایران نقش مهمی در تحولات اجتماعی و فرهنگی دارند. رشد جمعیت جوان و افزایش دسترسی به آموزش و فناوری‌های نوین باعث شده که نسل جدید با دیدگاه‌ها و مطالبه‌های متفاوت نسبت به گذشته وارد جامعه شود. این تغییرات در سبک زندگی، نگرش‌های سیاسی و فرهنگی و رفتارهای اجتماعی انعکاس یافته و باعث شکل‌گیری جریان‌های نوین فرهنگی و اجتماعی شده است.

افزایش آگاهی اجتماعی و سیاسی، استفاده از شبکه‌های اجتماعی و فعالیت‌های فرهنگی، فرصت‌هایی برای مشارکت و ابراز نظر فراهم کرده است. در عین حال، محدودیت‌های اقتصادی و اجتماعی، فشارهای فرهنگی و عدم تطابق برخی نهادهای سنتی با تغییرات جدید، باعث چالش‌هایی برای جوانان و خانواده‌ها شده است.

"""

In [7]:
def generate_questions(source_text, Prev_qust, num_questions=20):
    prompt = f"""
  تو یک طراح حرفه‌ای سؤال‌های آموزشی و آزمونی هستی.
بر اساس «متن منبع» که در ادامه آمده، سوالات چهارگزینه‌ای استاندارد از تاریخ معاصر ایران طراحی کن.

قوانین بسیار مهم:
1. همه سوالات باید به زبان فارسی باشند.
2. هر سوال باید دقیقاً چهار گزینه داشته باشد.
3. فقط یکی از گزینه‌ها پاسخ صحیح باشد.
4. گزینه صحیح حتماً باید عیناً یکی از گزینه‌ها باشد.
5. صورت سوال‌ها باید مفهومی، تحلیلی یا دانشی باشند (نه خیلی سطحی).
6. متن سوال‌ها باید کاملاً یونیک باشند و فقط با تغییر گزینه‌ها تکراری نشوند.
7. از تکرار مستقیم جملات متن منبع در صورت سوال خودداری کن.
8. سوالات باید متنوع باشند (سیاسی، اجتماعی، اقتصادی، فرهنگی).
9. اگر «سوالات قبلی» داده شد، مطلقاً نباید سوالی با مفهوم یا صورت مشابه تولید شود.
10. هیچ توضیح اضافی ننویس؛ فقط خروجی خواسته‌شده را بده.
11. از هر تیتر 8 سوال طراحی کن

 فرمت خروجی (فقط همین، بدون هیچ متن اضافی):
[
  {{
    "question": "متن سوال",
    "options": ["گزینه 1", "گزینه 2", "گزینه 3", "گزینه 4"],
    "answer": "گزینه صحیح"
  }}
]

 تعداد سوالات مورد نیاز: {num_questions}

 سوالات قبلی :'''
{Prev_qust}
'''


 متن منبع:
 '''
{source_text}
'''

اکنون دقیقاً مطابق قوانین بالا سوالات را تولید کن.

"""
    response = client.chat.completions.create(
    model="meta-llama/llama-4-maverick-17b-128e-instruct",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7
       )
    text = response.choices[0].message.content.strip()

    text = re.sub(r"```json|```", "", text)

    if not text:
        raise ValueError("خروجی مدل خالی بود!")

    return json.loads(text)

In [8]:
def previous_questions_text(all_questions):
    return "\n".join([q["question"] for q in all_questions])

In [9]:
questions = generate_questions(SOURCE_TEXT,'', num_questions=10)
print(questions)

[{'question': 'آقامحمدخان قاجار بنیان\u200cگذار کدام سلسله در ایران بود؟', 'options': ['صفویه', 'افشاریه', 'قاجار', 'پهلوی'], 'answer': 'قاجار'}, {'question': 'هدف اصلی آقامحمدخان قاجار در دوران حکومتش چه بود؟', 'options': ['اصلاحات اجتماعی و اقتصادی', 'بازسازی اقتدار سیاسی و تمامیت ارضی ایران', 'توسعه فرهنگی', 'گسترش حرمسرا'], 'answer': 'بازسازی اقتدار سیاسی و تمامیت ارضی ایران'}, {'question': 'فتحعلی\u200cشاه قاجار در دوران حکومتش با چه چالشی عمده\u200cای مواجه بود؟', 'options': ['جنگ\u200cهای داخلی', 'گسترش نفوذ قدرت\u200cهای استعماری روسیه و انگلستان', 'توسعه اقتصادی', 'اصلاحات اجتماعی'], 'answer': 'گسترش نفوذ قدرت\u200cهای استعماری روسیه و انگلستان'}, {'question': 'نتیجه جنگ\u200cهای ایران و روسیه در دوره فتحعلی\u200cشاه چه بود؟', 'options': ['پیروزی ایران و الحاق سرزمین\u200cهای جدید', 'انعقاد قراردادهای گلستان و ترکمانچای و جدایی بخش\u200cهایی از سرزمین\u200cهای قفقاز', 'صلح دائمی با روسیه', 'توسعه روابط اقتصادی با روسیه'], 'answer': 'انعقاد قراردادهای گلستان و ترکمانچای و جدایی

In [10]:
prev_text = "\n".join([q["question"] for q in questions])
print(prev_text)

آقامحمدخان قاجار بنیان‌گذار کدام سلسله در ایران بود؟
هدف اصلی آقامحمدخان قاجار در دوران حکومتش چه بود؟
فتحعلی‌شاه قاجار در دوران حکومتش با چه چالشی عمده‌ای مواجه بود؟
نتیجه جنگ‌های ایران و روسیه در دوره فتحعلی‌شاه چه بود؟
یکی از مهم‌ترین رویدادهای تاریخ معاصر ایران که در دوره مظفرالدین‌شاه رخ داد، چه بود؟
محمدعلی‌شاه قاجار چه رویکردی در قبال مشروطه داشت؟
رضاشاه پهلوی چگونه به قدرت رسید؟
انقلاب اسلامی ایران به چه نتیجه‌ای منجر شد؟
جنگ تحمیلی ایران و عراق چه تأثیری بر اقتصاد ایران داشت؟
یکی از چالش‌های اقتصادی ایران در دوره معاصر چیست؟


In [11]:
all_questions = []
  
SLEEP_BETWEEN_REQUESTS = 2
SIMILARITY_THRESHOLD = 0.7  
BATCH_SIZE = 20
TOTAL_QUESTIONS = 100

while len(all_questions) < TOTAL_QUESTIONS:
    prev_text = "\n".join([q["question"] for q in all_questions])
    candidate_questions = generate_questions(SOURCE_TEXT,prev_text,BATCH_SIZE)
    for q in candidate_questions:
        if isinstance(q, dict) and "question" in q:
            all_questions.append(q)
    print(f"سوال {len(all_questions)} اضافه شد")
    time.sleep(SLEEP_BETWEEN_REQUESTS)


سوال 20 اضافه شد
سوال 40 اضافه شد
سوال 60 اضافه شد
سوال 80 اضافه شد
سوال 100 اضافه شد


In [12]:
with open("all_questions.json", "w", encoding="utf-8") as f:
    json.dump(all_questions, f, ensure_ascii=False, indent=2)


In [3]:
with open("/kaggle/input/question/all_questions.json", "r", encoding="utf-8") as f:
    all_questions = json.load(f)

In [48]:
all_questions[2]

{'question': 'چرا حکومت آقامحمدخان بیشتر به عنوان دولت نظامی متمرکز شناخته می\u200cشود؟',
 'options': ['تمرکز بر اصلاحات اجتماعی',
  'اولویت تثبیت قدرت سیاسی',
  'گسترش نهادهای مدنی',
  'توسعه اقتصادی گسترده'],
 'answer': 'اولویت تثبیت قدرت سیاسی'}

In [4]:
train_data, temp_data = train_test_split(all_questions, test_size=0.3, random_state=42)
val_data, test_data   = train_test_split(temp_data, test_size=2/3, random_state=42)

with open("train.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
with open("validation.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)
with open("test.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)


In [16]:
train_data[2]

{'question': 'چرا نارضایتی اجتماعی دوره محمدشاه به جنبش سیاسی تبدیل نشد؟',
 'options': ['نبود فشار خارجی',
  'فقدان سازمان\u200cیافتگی سیاسی',
  'حمایت کامل مردم از شاه',
  'اصلاحات گسترده'],
 'answer': 'فقدان سازمان\u200cیافتگی سیاسی'}

In [23]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LEN = 128
OUTPUT_DIR = "tinyllama-lora-emotion"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_auth_token=token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, use_auth_token=token)
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py:1025: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [15]:
MAX_LEN = 256

def preprocess(example):
    option_letters = ["A", "B", "C", "D"]

    prompt = f"Question: {example['question']}\nOptions:\n"
    for i, opt in enumerate(example["options"]):
        prompt += f"{option_letters[i]}) {opt}\n"
 
    answer_letter = option_letters[
        example["options"].index(example["answer"])
    ]

    full_text = prompt + f"Answer: {answer_letter}"

    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
    
train_dataset = Dataset.from_list(train_data).map(preprocess) 
val_dataset = Dataset.from_list(val_data).map(preprocess) 
test_dataset = Dataset.from_list(test_data).map(preprocess)

dataset = DatasetDict({ "train" : train_dataset, "validation": val_dataset , "test" : test_dataset })

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

In [18]:
import torch
import torch.nn.functional as F

def predict_answer(example):
    model.eval()
    device = next(model.parameters()).device

    option_letters = ["A", "B", "C", "D"]

    prompt = f"Question: {example['question']}\nOptions:\n"
    for i, opt in enumerate(example["options"]):
        prompt += f"{option_letters[i]}) {opt}\n"
    prompt += "Answer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1] 

    log_probs = F.log_softmax(logits, dim=-1)

    scores = {}
    for letter in option_letters[:len(example["options"])]:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        scores[letter] = log_probs[token_id].item()

    best_letter = max(scores, key=scores.get)

    return best_letter, scores

In [19]:
def evaluate_model(model, dataset):
    correct = 0

    for example in dataset:
        pred_letter, scores = predict_answer(example)

        true_letter = ["A", "B", "C", "D"][
            example["options"].index(example["answer"])
        ]

        if pred_letter == true_letter:
            correct += 1

    return correct / len(dataset)

accuracy = evaluate_model(model, dataset["test"])
print(f"Before Train Accuracy: {accuracy*100:.2f}%")

Before Train Accuracy: 69.23%


In [20]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [23]:
print(len(dataset["train"]),"\n",len(dataset["validation"]),"\n",len(dataset["test"]))

42 
 6 
 13


In [24]:
print(train_dataset[2])

{'question': 'چرا نارضایتی اجتماعی دوره محمدشاه به جنبش سیاسی تبدیل نشد؟', 'options': ['نبود فشار خارجی', 'فقدان سازمان\u200cیافتگی سیاسی', 'حمایت کامل مردم از شاه', 'اصلاحات گسترده'], 'answer': 'فقدان سازمان\u200cیافتگی سیاسی', 'input_ids': [1, 894, 29901, 29871, 30905, 30156, 30112, 29871, 30162, 30112, 30156, 30624, 30112, 30202, 30195, 30202, 29871, 30112, 30270, 30195, 30159, 30112, 30218, 30202, 29871, 30172, 30171, 30156, 30204, 29871, 30159, 30240, 30159, 30172, 30256, 30112, 30204, 29871, 30177, 30204, 29871, 30270, 30162, 30177, 30256, 29871, 30198, 30202, 30112, 30198, 30202, 29871, 30195, 30177, 30172, 30202, 30138, 29871, 30162, 30256, 30172, 219, 162, 13, 5856, 29901, 13, 29909, 29897, 29871, 30162, 30177, 30171, 30172, 29871, 30241, 30256, 30112, 30156, 29871, 30321, 30112, 30156, 30270, 30202, 13, 29933, 29897, 29871, 30241, 30265, 30172, 30112, 30162, 29871, 30198, 30112, 30295, 30159, 30112, 30162, 30430, 30202, 30112, 30241, 30195, 30421, 30202, 29871, 30198, 30202, 

In [21]:
from transformers import TrainingArguments, Trainer, default_data_collator

training_args = TrainingArguments(
    output_dir="./lora_finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    eval_steps=500,
    save_steps=500,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=default_data_collator,
)


Step,Training Loss


TrainOutput(global_step=18, training_loss=4.759249793158637, metrics={'train_runtime': 22.365, 'train_samples_per_second': 5.634, 'train_steps_per_second': 0.805, 'total_flos': 200433387700224.0, 'train_loss': 4.759249793158637, 'epoch': 3.0})

In [24]:
print("=== Eval metrics BEFORE training ===")
metrics_before = trainer.evaluate()
print(metrics_before)

print("===  metrics BEFORE training ===")
cls_before = evaluate_model(model, dataset["test"])
print(cls_before)

=== Eval metrics BEFORE training ===


{'eval_loss': 1.3568071126937866, 'eval_runtime': 0.5225, 'eval_samples_per_second': 11.484, 'eval_steps_per_second': 1.914, 'epoch': 3.0}
===  metrics BEFORE training ===
0.6923076923076923


In [25]:

trainer.train()

Step,Training Loss


TrainOutput(global_step=18, training_loss=0.944732560051812, metrics={'train_runtime': 22.2748, 'train_samples_per_second': 5.657, 'train_steps_per_second': 0.808, 'total_flos': 200433387700224.0, 'train_loss': 0.944732560051812, 'epoch': 3.0})

In [26]:
print("=== Eval metrics AFTER training ===")
metrics_before = trainer.evaluate()
print(metrics_before)

print("===  metrics AFTER training ===")
cls_before = evaluate_model(model, dataset["test"])
print(cls_before)

=== Eval metrics AFTER training ===
{'eval_loss': 0.8680155873298645, 'eval_runtime': 0.5226, 'eval_samples_per_second': 11.482, 'eval_steps_per_second': 1.914, 'epoch': 3.0}
===  metrics AFTER training ===
0.6923076923076923
